# Tokenization

This notebook explores how raw text is converted into tokens and token IDs.

Topics covered:
- Character-level tokenization
- Byte Pair Encoding (BPE)
- Learning BPE merge rules
- Applying learned merges to new text
- Special tokens
- Padding
- Attention masks

In [134]:
from collections import Counter

In [135]:
text="low lower lowest low lower"
words=text.split()
token=[]
for word in words:
    token.append([i for i in word]+["</w>"])
print("Tokens in each words:",token)

Tokens in each words: [['l', 'o', 'w', '</w>'], ['l', 'o', 'w', 'e', 'r', '</w>'], ['l', 'o', 'w', 'e', 's', 't', '</w>'], ['l', 'o', 'w', '</w>'], ['l', 'o', 'w', 'e', 'r', '</w>']]


### Merge Function

In [136]:
def merge(common,token):
    i=0
    new_token=[]
    while i<len(token):
        if i<len(token)-1 and (token[i],token[i+1])==common:
            new_token.append("".join(common))
            i+=2
        else:
            new_token.append(token[i])
            i+=1
    return new_token

### Pairing and Combining Frequent Tokens

In [137]:
for i in range(2):
    pairs=[]
    for i in token:
        for j in range(len(i)-1):
            pairs.append((i[j],i[j+1]))
    adjacent_frequency=Counter(pairs)
    print("\n\nFrequency of adjacent tokens:",adjacent_frequency)
    most_common=adjacent_frequency.most_common(1)[0][0]
    print("Most Common pair:",most_common)
    for i in range(len(token)):
        token[i]=merge(most_common,token[i])
print("\nFinal vocab tokens:",token)



Frequency of adjacent tokens: Counter({('l', 'o'): 5, ('o', 'w'): 5, ('w', 'e'): 3, ('w', '</w>'): 2, ('e', 'r'): 2, ('r', '</w>'): 2, ('e', 's'): 1, ('s', 't'): 1, ('t', '</w>'): 1})
Most Common pair: ('l', 'o')


Frequency of adjacent tokens: Counter({('lo', 'w'): 5, ('w', 'e'): 3, ('w', '</w>'): 2, ('e', 'r'): 2, ('r', '</w>'): 2, ('e', 's'): 1, ('s', 't'): 1, ('t', '</w>'): 1})
Most Common pair: ('lo', 'w')

Final vocab tokens: [['low', '</w>'], ['low', 'e', 'r', '</w>'], ['low', 'e', 's', 't', '</w>'], ['low', '</w>'], ['low', 'e', 'r', '</w>']]


### Splitting of new words

In [138]:
new_word="lowest"
new_tokens=list(new_word)+["</w>"]
for pair in token:
    new_tokens=merge(new_tokens,pair)
print(new_tokens)

['low', 'e', 'r', '</w>']


### Add Special Tokens to vocab

In [139]:
text1="the cat"
text2="the dog jumped"
text1=text1.split()
text2=text2.split()
special_token=["<UNK>","<PAD>","<BOS>","<EOS>"]
text=text1+text2
vocab=list(set(text))

word2idx={}
idx2word={}
for idx,val in enumerate (vocab):
    word2idx[val]=idx
for i in special_token:
    if i not in word2idx:
        word2idx[i]=len(word2idx)
for val,idx in word2idx.items():
    idx2word[idx]=val
print(word2idx)

{'jumped': 0, 'cat': 1, 'the': 2, 'dog': 3, '<UNK>': 4, '<PAD>': 5, '<BOS>': 6, '<EOS>': 7}


### UNK and Special Token to new words

In [140]:
new_words=["the cat and dog jumped together","the cat loves playing","dog is naughty"]
batch=[]
for new_word in new_words:
    new_word=["<BOS>"]+new_word.split()+["<EOS>"]
    token=[]
    for word in new_word:
        token.append(word2idx.get(word,word2idx["<UNK>"]))
    batch.append(token)
batch

[[6, 2, 1, 4, 3, 0, 4, 7], [6, 2, 1, 4, 4, 7], [6, 3, 4, 4, 7]]

### Padding sequences in batch

In [141]:
maximum=max(map(lambda x:len(x),batch))
for i in range(len(batch)):
    batch[i].pop()
    for j in range(maximum-1-len(batch[i])):
        batch[i].append(word2idx["<PAD>"])
    batch[i].append(word2idx["<EOS>"])
batch

[[6, 2, 1, 4, 3, 0, 4, 7], [6, 2, 1, 4, 4, 5, 5, 7], [6, 3, 4, 4, 5, 5, 5, 7]]

### Masked Padding

In [142]:
batchmask=[[token!=word2idx["<PAD>"]for token in sequence]for sequence in batch]
batchmask

[[True, True, True, True, True, True, True, True],
 [True, True, True, True, True, False, False, True],
 [True, True, True, True, False, False, False, True]]